# 02. Limpieza y Enriquecimiento de Datos (Feature Engineering)

Una vez que entendieron los datos, vamos a prepararlos para que los algoritmos de Machine Learning puedan procesarlos correctamente. En la vida real, lo mejor es empaquetar esto automatizado en Scikit-Learn Pipelines o en scripts de Python (`src/features/build_features.py`). Aquí puedes experimentar con las rutinas de limpieza.

### Instrucciones Generales:
1. **Solvertar problema de calidad:**: Solucionar problema de calidad encontrados en el EDA: consistencia, sensibilidad, precision y completitud. Documenta cada decision tomada.
2. **Codificación Categórica:** El campo `ocean_proximity` es de texto. Conviértelo en variable numerica, ya que los algoritmos clasicos no entienen el texto. Documenta porque usaste codificacion Ordinal o Nominal.
3. **Enriquecimiento (Feature Engineering):** Como pudiste notar en tu análisis, `total_rooms` no significa mucho si hay muchos hogares en un distrito. Agrega nuevas métricas útiles, por ejemplo:
   - `rooms_per_household = total_rooms / households`
   - `bedrooms_per_room = total_bedrooms / total_rooms`
   - `population_per_household = population / households`
4. **Escalado de Variables:** Aplica un `StandardScaler` o `MinMaxScaler` para evitar que las variables numéricas grandes pesen más en algoritmos basados en distancias o gradientes.


In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# Cargar datos
df = pd.read_csv("../data/interim/train_set.csv")

# Copia de trabajo
df_clean = df.copy()

print("Dimensión original:", df_clean.shape)
df_clean.head()

Dimensión original: (16512, 10)


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.42,37.80,52.0,3321.0,1115.0,1576.0,1034.0,2.0987,458300.0,NEAR BAY
1,-118.38,34.14,40.0,1965.0,354.0,666.0,357.0,6.0876,483800.0,<1H OCEAN
2,-121.98,38.36,33.0,1083.0,217.0,562.0,203.0,2.4330,101700.0,INLAND
3,-117.11,33.75,17.0,4174.0,851.0,1845.0,780.0,2.2618,96100.0,INLAND
4,-118.15,33.77,36.0,4366.0,1211.0,1912.0,1172.0,3.5292,361800.0,NEAR OCEAN


In [2]:
# Calidad inicial
print("Tipos de datos:")
print(df_clean.dtypes)

print("\nFaltantes por columna:")
print(df_clean.isnull().sum().sort_values(ascending=False))

print("\nResumen estadístico:")
display(df_clean.describe(include="all").T)

Tipos de datos:
longitude             float64
latitude              float64
housing_median_age    float64
total_rooms           float64
total_bedrooms        float64
population            float64
households            float64
median_income         float64
median_house_value    float64
ocean_proximity        object
dtype: object

Faltantes por columna:
total_bedrooms        168
longitude               0
latitude                0
housing_median_age      0
total_rooms             0
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64

Resumen estadístico:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
longitude,16512.0,NaN,NaN,NaN,-119.573125,2.000624,-124.35,-121.8,-118.51,-118.01,-114.49
latitude,16512.0,NaN,NaN,NaN,35.637746,2.133294,32.55,33.93,34.26,37.72,41.95
housing_median_age,16512.0,NaN,NaN,NaN,28.577156,12.585738,1.0,18.0,29.0,37.0,52.0
total_rooms,16512.0,NaN,NaN,NaN,2639.402798,2185.287466,2.0,1447.0,2125.0,3154.0,39320.0
total_bedrooms,16344.0,NaN,NaN,NaN,538.949094,423.862079,1.0,296.0,434.0,645.0,6210.0
population,16512.0,NaN,NaN,NaN,1425.513929,1094.795467,3.0,787.0,1167.0,1726.0,16305.0
households,16512.0,NaN,NaN,NaN,499.990189,382.865787,1.0,279.0,408.0,603.0,5358.0
median_income,16512.0,NaN,NaN,NaN,3.870428,1.891936,0.4999,2.5625,3.5385,4.75,15.0001
median_house_value,16512.0,NaN,NaN,NaN,206333.518653,115314.047529,14999.0,119200.0,179200.0,263925.0,500001.0
ocean_proximity,16512,5,<1H OCEAN,7274,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# Medir asimetría de las variables numéricas
skewness = df_clean.select_dtypes(include="number").skew().sort_values(ascending=False)
print(skewness)

total_rooms           4.163762
total_bedrooms        3.444447
households            3.357607
population            3.283510
median_income         1.617601
median_house_value    0.987685
latitude              0.448789
housing_median_age    0.061423
longitude            -0.287729
dtype: float64


### Completitud

In [4]:
# Revisar faltantes antes de limpiar
faltantes_abs = df_clean.isnull().sum().sort_values(ascending=False)
faltantes_pct = (df_clean.isnull().sum() / len(df_clean) * 100).sort_values(ascending=False)

print("Faltantes absolutos:")
print(faltantes_abs)

print("\nFaltantes porcentuales:")
print(faltantes_pct.round(2))

Faltantes absolutos:
total_bedrooms        168
longitude               0
latitude                0
housing_median_age      0
total_rooms             0
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64

Faltantes porcentuales:
total_bedrooms        1.02
longitude             0.00
latitude              0.00
housing_median_age    0.00
total_rooms           0.00
population            0.00
households            0.00
median_income         0.00
median_house_value    0.00
ocean_proximity       0.00
dtype: float64


In [5]:
#En el EDA se identificó que la única variable con faltantes es total_bedrooms, con aproximadamente 1% de registros incompletos. 
# No se eliminan esas filas porque, aunque el porcentaje es pequeño, cada registro sigue conteniendo información valiosa en variables
# relevantes como median_income, population, households, latitude, longitude y ocean_proximity

In [6]:
# Se imputa total_bedrooms usando relacion total_bedrooms / total_rooms

# Crear razón auxiliar solo en filas válidas
mask_validas = df_clean["total_bedrooms"].notna() & (df_clean["total_rooms"] > 0)

df_clean.loc[mask_validas, "bedrooms_per_room_aux"] = (
    df_clean.loc[mask_validas, "total_bedrooms"] / df_clean.loc[mask_validas, "total_rooms"]
)

# Mediana global de respaldo
mediana_global_ratio = df_clean["bedrooms_per_room_aux"].median()

# Mediana por grupo geográfico
mediana_ratio_por_zona = (
    df_clean.loc[mask_validas]
    .groupby("ocean_proximity")["bedrooms_per_room_aux"]
    .median()
)

print("Mediana global bedrooms_per_room:", round(mediana_global_ratio, 4))
print("\nMediana bedrooms_per_room por ocean_proximity:")
print(mediana_ratio_por_zona)

# Imputación basada en estructura del dato
mask_missing = df_clean["total_bedrooms"].isna()

df_clean.loc[mask_missing, "total_bedrooms"] = (
    df_clean.loc[mask_missing, "total_rooms"] *
    df_clean.loc[mask_missing, "ocean_proximity"].map(mediana_ratio_por_zona).fillna(mediana_global_ratio)
).round()

# Regla de seguridad lógica
df_clean["total_bedrooms"] = df_clean["total_bedrooms"].clip(lower=1)
df_clean["total_bedrooms"] = np.minimum(df_clean["total_bedrooms"], df_clean["total_rooms"])

# Eliminar auxiliar
df_clean.drop(columns=["bedrooms_per_room_aux"], inplace=True)

print("\nFaltantes después de imputar:")
print(df_clean.isnull().sum())

Mediana global bedrooms_per_room: 0.2031

Mediana bedrooms_per_room por ocean_proximity:
ocean_proximity
<1H OCEAN     0.206428
INLAND        0.198287
ISLAND        0.288053
NEAR BAY      0.206871
NEAR OCEAN    0.207420
Name: bedrooms_per_room_aux, dtype: float64

Faltantes después de imputar:
longitude             0
latitude              0
housing_median_age    0
total_rooms           0
total_bedrooms        0
population            0
households            0
median_income         0
median_house_value    0
ocean_proximity       0
dtype: int64


Sustento:
No se usó una mediana global fija de total_bedrooms, porque esa solución puede generar incoherencias en distritos pequeños, por ejemplo casos donde el número imputado de dormitorios supere al total de habitaciones. En su lugar, se imputó usando la relación total_bedrooms / total_rooms, calculada por grupo de ocean_proximity. Esta decisión es más sólida porque preserva la estructura real del dato: el número de dormitorios depende del tamaño del distrito y del tipo de zona geográfica

## Consistencia y precisión

In [7]:
# Normalizar variable categórica y revisar duplicados
# Normalizar texto
df_clean["ocean_proximity"] = (
    df_clean["ocean_proximity"]
    .astype(str)
    .str.strip()
    .str.upper()
)

print("Valores únicos de ocean_proximity:")
print(df_clean["ocean_proximity"].value_counts(dropna=False))

# Revisar duplicados exactos
duplicados = df_clean.duplicated().sum()
print("\nNúmero de duplicados exactos:", duplicados)

if duplicados > 0:
    df_clean = df_clean.drop_duplicates().copy()
    print("Duplicados eliminados.")
else:
    print("No se encontraron duplicados exactos.")

Valores únicos de ocean_proximity:
ocean_proximity
<1H OCEAN     7274
INLAND        5301
NEAR OCEAN    2089
NEAR BAY      1846
ISLAND           2
Name: count, dtype: int64

Número de duplicados exactos: 0
No se encontraron duplicados exactos.


In [8]:
# Validar reglas lógicas del negocio

valores_validos_ocean = {"<1H OCEAN", "INLAND", "NEAR OCEAN", "NEAR BAY", "ISLAND"}

# Reglas lógicas
condicion_valida = (
    (df_clean["ocean_proximity"].isin(valores_validos_ocean)) &
    (df_clean["total_bedrooms"] <= df_clean["total_rooms"]) &
    (df_clean["households"] <= df_clean["population"]) &
    (df_clean["total_rooms"] > 0) &
    (df_clean["total_bedrooms"] > 0) &
    (df_clean["population"] > 0) &
    (df_clean["households"] > 0) &
    (df_clean["median_income"] > 0) &
    (df_clean["latitude"] >= 32) & (df_clean["latitude"] <= 42) &
    (df_clean["longitude"] >= -125) & (df_clean["longitude"] <= -114)
)

df_invalidos = df_clean.loc[~condicion_valida].copy()

print("Registros inválidos detectados:", df_invalidos.shape[0])
print("\nPrimeros registros inválidos:")
print(df_invalidos.head())

# Eliminar registros inválidos
df_clean = df_clean.loc[condicion_valida].copy()

print("\nDimensión después de eliminar inválidos:", df_clean.shape)

Registros inválidos detectados: 3

Primeros registros inválidos:
       longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
6419     -121.00     39.75                 8.0       1116.0           214.0   
7162     -118.44     34.04                16.0         18.0             6.0   
14909    -121.00     37.65                17.0        484.0           202.0   

       population  households  median_income  median_house_value  \
6419         27.0        39.0         2.5893             83000.0   
7162          3.0         4.0         0.5360            350000.0   
14909       198.0       204.0         0.6825            187500.0   

      ocean_proximity  
6419           INLAND  
7162        <1H OCEAN  
14909          INLAND  

Dimensión después de eliminar inválidos: (16509, 10)


Aquí no se eliminaron outliers, sino registros lógicamente imposibles. Por ejemplo, total_bedrooms no puede ser mayor que total_rooms, y households no debería superar a population. Este tipo de registros no representan variabilidad real del negocio, sino inconsistencias del dato. Por eso sí deben eliminarse para no introducir ruido en el modelado.

## Sensibilidad

In [9]:
#Cuantificar valores extremos
def resumen_outliers_iqr(df_in, columnas):
    resumen = []
    for col in columnas:
        q1 = df_in[col].quantile(0.25)
        q3 = df_in[col].quantile(0.75)
        iqr = q3 - q1
        lim_inf = q1 - 1.5 * iqr
        lim_sup = q3 + 1.5 * iqr
        n_out = ((df_in[col] < lim_inf) | (df_in[col] > lim_sup)).sum()
        pct_out = round(n_out / len(df_in) * 100, 2)
        resumen.append({
            "variable": col,
            "Q1": q1,
            "Q3": q3,
            "IQR": iqr,
            "lim_inf": lim_inf,
            "lim_sup": lim_sup,
            "n_outliers": n_out,
            "pct_outliers": pct_out
        })
    return pd.DataFrame(resumen)

columnas_sensibles = [
    "total_rooms", "total_bedrooms", "population", "households", "median_income"
]

resumen_outliers = resumen_outliers_iqr(df_clean, columnas_sensibles)
print(resumen_outliers)

         variable         Q1       Q3        IQR     lim_inf     lim_sup  \
0     total_rooms  1448.0000  3154.00  1706.0000 -1111.00000  5713.00000   
1  total_bedrooms   295.0000   646.00   351.0000  -231.50000  1172.50000   
2      population   787.0000  1726.00   939.0000  -621.50000  3134.50000   
3      households   280.0000   603.00   323.0000  -204.50000  1087.50000   
4   median_income     2.5625     4.75     2.1875    -0.71875     8.03125   

   n_outliers  pct_outliers  
0        1041          6.31  
1        1045          6.33  
2         969          5.87  
3        1003          6.08  
4         537          3.25  


Se incluyó median_income en la revisión de outliers porque en el EDA mostró una cola derecha importante y una relación fuerte con la variable objetivo. No revisarlo habría sido inconsistente. La detección se hizo con IQR para identificar extremos, pero no se decidió eliminarlos automáticamente, ya que en este problema muchos valores altos pueden ser reales y reflejar distritos costeros caros o zonas densamente pobladas.

In [10]:
# Mantener outliers y estabilizar su efecto
for col in columnas_sensibles:
    df_clean[f"log_{col}"] = np.log1p(df_clean[col])

print("Variables logarítmicas creadas:")
print([f"log_{col}" for col in columnas_sensibles])

Variables logarítmicas creadas:
['log_total_rooms', 'log_total_bedrooms', 'log_population', 'log_households', 'log_median_income']


Sustento: Los outliers se conservan porque forman parte de la realidad del mercado inmobiliario. Eliminar valores altos de ingresos, población o habitaciones podría borrar señal importante del negocio. En vez de recortar o borrar registros reales, se crean transformaciones logarítmicas para reducir asimetría y disminuir el peso excesivo de los extremos en modelos sensibles.

## Codificación categórica

In [11]:
# Transformar ocean_proximity a formato numérico

df_fe = pd.get_dummies(
    df_clean,
    columns=["ocean_proximity"],
    prefix="ocean",
    dtype=int
)

print("Columnas después de codificación:")
print(df_fe.columns.tolist())
df_fe.head()

Columnas después de codificación:
['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value', 'log_total_rooms', 'log_total_bedrooms', 'log_population', 'log_households', 'log_median_income', 'ocean_<1H OCEAN', 'ocean_INLAND', 'ocean_ISLAND', 'ocean_NEAR BAY', 'ocean_NEAR OCEAN']


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,log_total_rooms,log_total_bedrooms,log_population,log_households,log_median_income,ocean_<1H OCEAN,ocean_INLAND,ocean_ISLAND,ocean_NEAR BAY,ocean_NEAR OCEAN
0,-122.42,37.80,52.0,3321.0,1115.0,1576.0,1034.0,2.0987,458300.0,8.108322,7.017506,7.363280,6.942157,1.130983,0,0,0,1,0
1,-118.38,34.14,40.0,1965.0,354.0,666.0,357.0,6.0876,483800.0,7.583756,5.872118,6.502790,5.880533,1.958347,1,0,0,0,0
2,-121.98,38.36,33.0,1083.0,217.0,562.0,203.0,2.4330,101700.0,6.988413,5.384495,6.333280,5.318120,1.233435,0,1,0,0,0
3,-117.11,33.75,17.0,4174.0,851.0,1845.0,780.0,2.2618,96100.0,8.336870,6.747587,7.520776,6.660575,1.182279,0,1,0,0,0
4,-118.15,33.77,36.0,4366.0,1211.0,1912.0,1172.0,3.5292,361800.0,8.381832,7.100027,7.556428,7.067320,1.510545,0,0,0,0,1


Sustento: Se usó One-Hot Encoding y no codificación ordinal porque ocean_proximity es una variable nominal, no ordinal. Es decir, sus categorías representan tipos de ubicación y no tienen un orden natural. Asignar valores como 1, 2, 3 o 4 podría inducir al modelo a interpretar una jerarquía que no existe.

## Enriquecimiento de datos (Feature Engineering)


In [12]:
#Crear variables derivadas más útiles que los totales brutos
df_fe["rooms_per_household"] = df_fe["total_rooms"] / df_fe["households"]
df_fe["bedrooms_per_room"] = df_fe["total_bedrooms"] / df_fe["total_rooms"]
df_fe["population_per_household"] = df_fe["population"] / df_fe["households"]
df_fe["bedrooms_per_household"] = df_fe["total_bedrooms"] / df_fe["households"]
df_fe["rooms_per_person"] = df_fe["total_rooms"] / df_fe["population"]

nuevas_features = [
    "rooms_per_household",
    "bedrooms_per_room",
    "population_per_household",
    "bedrooms_per_household",
    "rooms_per_person"
]

print(df_fe[nuevas_features].describe())

       rooms_per_household  bedrooms_per_room  population_per_household  \
count         16509.000000       16509.000000              16509.000000   
mean              5.439849           0.212690                  2.996373   
std               2.567929           0.057107                  4.457679   
min               0.888889           0.100000                  1.066176   
25%               4.443640           0.175560                  2.433657   
50%               5.235632           0.203350                  2.822819   
75%               6.053691           0.238903                  3.286385   
max             141.909091           1.000000                502.461538   

       bedrooms_per_household  rooms_per_person  
count            16509.000000      16509.000000  
mean                 1.098029          1.977416  
std                  0.497005          1.146791  
min                  0.333333          0.018065  
25%                  1.006029          1.521974  
50%                  1.0

Las variables absolutas como total_rooms, population o households describen tamaño, pero no capturan densidad ni composición del distrito. Por eso se crean métricas derivadas que son más interpretables:

rooms_per_household: aproxima disponibilidad de habitaciones por hogar.
bedrooms_per_room: mide la proporción de dormitorios respecto al total de habitaciones.
population_per_household: aproxima tamaño medio del hogar.
bedrooms_per_household: relaciona dormitorios con número de hogares.
rooms_per_person: aproxima espacio habitacional por persona.

Estas variables son más útiles para modelar valor de vivienda porque representan condiciones estructurales del distrito y no solo magnitudes brutas.

In [13]:
#Controlar extremos en variables derivadas
for col in nuevas_features:
    p1 = df_fe[col].quantile(0.01)
    p99 = df_fe[col].quantile(0.99)
    df_fe[col] = df_fe[col].clip(lower=p1, upper=p99)
    print(f"{col}: clip entre P1={round(p1, 4)} y P99={round(p99, 4)}")

rooms_per_household: clip entre P1=2.6091 y P99=10.5127
bedrooms_per_room: clip entre P1=0.1247 y P99=0.4046
population_per_household: clip entre P1=1.5515 y P99=5.423
bedrooms_per_household: clip entre P1=0.8572 y P99=2.1583
rooms_per_person: clip entre P1=0.6567 y P99=4.3721


Las variables derivadas tipo razón pueden explotar cuando el denominador es pequeño, aunque el registro siga siendo válido. Por eso, en lugar de eliminar observaciones, se aplica winsorización suave entre percentil 1 y 99. Esta decisión reduce la sensibilidad a extremos artificiales sin destruir información del dataset.

In [14]:
pd.set_option("display.max_columns", None)

df_fe.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,log_total_rooms,log_total_bedrooms,log_population,log_households,log_median_income,ocean_<1H OCEAN,ocean_INLAND,ocean_ISLAND,ocean_NEAR BAY,ocean_NEAR OCEAN,rooms_per_household,bedrooms_per_room,population_per_household,bedrooms_per_household,rooms_per_person
0,-122.42,37.80,52.0,3321.0,1115.0,1576.0,1034.0,2.0987,458300.0,8.108322,7.017506,7.363280,6.942157,1.130983,0,0,0,1,0,3.211799,0.335742,1.551466,1.078337,2.107234
1,-118.38,34.14,40.0,1965.0,354.0,666.0,357.0,6.0876,483800.0,7.583756,5.872118,6.502790,5.880533,1.958347,1,0,0,0,0,5.504202,0.180153,1.865546,0.991597,2.950450
2,-121.98,38.36,33.0,1083.0,217.0,562.0,203.0,2.4330,101700.0,6.988413,5.384495,6.333280,5.318120,1.233435,0,1,0,0,0,5.334975,0.200369,2.768473,1.068966,1.927046
3,-117.11,33.75,17.0,4174.0,851.0,1845.0,780.0,2.2618,96100.0,8.336870,6.747587,7.520776,6.660575,1.182279,0,1,0,0,0,5.351282,0.203881,2.365385,1.091026,2.262331
4,-118.15,33.77,36.0,4366.0,1211.0,1912.0,1172.0,3.5292,361800.0,8.381832,7.100027,7.556428,7.067320,1.510545,0,0,0,0,1,3.725256,0.277371,1.631399,1.033276,2.283473


## Escalado de variables

In [15]:
# Identificar variables que sí deben escalarse

# Variable objetivo
target = "median_house_value"

# Variables dummy creadas a partir de ocean_proximity
columnas_dummy = [col for col in df_fe.columns if col.startswith("ocean_")]

# Variables numéricas predictoras
columnas_numericas = df_fe.drop(columns=[target]).select_dtypes(include=np.number).columns.tolist()

# Variables continuas que sí deben escalarse
columnas_continuas = [col for col in columnas_numericas if col not in columnas_dummy]

print("Columnas dummy (no se escalan):")
print(columnas_dummy)

print("\nColumnas continuas a escalar:")
print(columnas_continuas)

print("\nCantidad de columnas en df_fe:", df_fe.shape[1])

Columnas dummy (no se escalan):
['ocean_<1H OCEAN', 'ocean_INLAND', 'ocean_ISLAND', 'ocean_NEAR BAY', 'ocean_NEAR OCEAN']

Columnas continuas a escalar:
['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'log_total_rooms', 'log_total_bedrooms', 'log_population', 'log_households', 'log_median_income', 'rooms_per_household', 'bedrooms_per_room', 'population_per_household', 'bedrooms_per_household', 'rooms_per_person']

Cantidad de columnas en df_fe: 24


Se escalan únicamente las variables numéricas continuas porque son las que pueden dominar el entrenamiento por sus diferentes magnitudes. No se escala median_house_value porque es la variable objetivo y conviene mantenerla en su unidad original para interpretar mejor las predicciones. Tampoco se escalan las variables dummy de ocean_proximity, ya que ya representan presencia o ausencia en formato 0/1.

In [16]:
#Aplicar StandardScaler a las variables continuas

# Crear una copia para no sobrescribir df_fe
df_fe_scaled = df_fe.copy()

# Inicializar escalador
scaler = StandardScaler()

# Escalar solo variables continuas
df_fe_scaled[columnas_continuas] = scaler.fit_transform(df_fe[columnas_continuas])

print("Escalado aplicado correctamente.")
print("\nShape original:", df_fe.shape)
print("Shape escalado:", df_fe_scaled.shape)

Escalado aplicado correctamente.

Shape original: (16509, 24)
Shape escalado: (16509, 24)


Se utiliza StandardScaler porque los modelos lineales y basados en gradiente, como LinearRegression y especialmente SGDRegressor, funcionan mejor cuando las variables están centradas y en escalas comparables

In [17]:
#Verificar que el escalado se aplicó solo donde corresponde
# Resumen estadístico de las variables escaladas
resumen_escalado = pd.DataFrame({
    "media_aprox": df_fe_scaled[columnas_continuas].mean(),
    "std_aprox": df_fe_scaled[columnas_continuas].std(ddof=0)
}).round(3)

print("Resumen de variables continuas escaladas:")
display(resumen_escalado.head(15))

# Verificar que target no cambió
print("¿La variable objetivo quedó igual?:",
      df_fe[target].equals(df_fe_scaled[target]))

# Verificar que las dummies no cambiaron
dummies_iguales = all(df_fe[col].equals(df_fe_scaled[col]) for col in columnas_dummy)
print("¿Las variables dummy quedaron iguales?:", dummies_iguales)

Resumen de variables continuas escaladas:


,media_aprox,std_aprox
longitude,0.0,1.0
latitude,-0.0,1.0
housing_median_age,0.0,1.0
total_rooms,0.0,1.0
total_bedrooms,0.0,1.0
population,0.0,1.0
households,-0.0,1.0
median_income,0.0,1.0
log_total_rooms,-0.0,1.0
log_total_bedrooms,0.0,1.0


¿La variable objetivo quedó igual?: True
¿Las variables dummy quedaron iguales?: True


In [18]:
# Comparar una muestra antes y después del escalado
print("Vista de df_fe (sin escalar):")
display(df_fe.head(3))

print("Vista de df_fe_scaled (escalado):")
display(df_fe_scaled.head(3))

Vista de df_fe (sin escalar):


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,log_total_rooms,log_total_bedrooms,log_population,log_households,log_median_income,ocean_<1H OCEAN,ocean_INLAND,ocean_ISLAND,ocean_NEAR BAY,ocean_NEAR OCEAN,rooms_per_household,bedrooms_per_room,population_per_household,bedrooms_per_household,rooms_per_person
0,-122.42,37.80,52.0,3321.0,1115.0,1576.0,1034.0,2.0987,458300.0,8.108322,7.017506,7.36328,6.942157,1.130983,0,0,0,1,0,3.211799,0.335742,1.551466,1.078337,2.107234
1,-118.38,34.14,40.0,1965.0,354.0,666.0,357.0,6.0876,483800.0,7.583756,5.872118,6.50279,5.880533,1.958347,1,0,0,0,0,5.504202,0.180153,1.865546,0.991597,2.950450
2,-121.98,38.36,33.0,1083.0,217.0,562.0,203.0,2.4330,101700.0,6.988413,5.384495,6.33328,5.318120,1.233435,0,1,0,0,0,5.334975,0.200369,2.768473,1.068966,1.927046


Vista de df_fe_scaled (escalado):


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,log_total_rooms,log_total_bedrooms,log_population,log_households,log_median_income,ocean_<1H OCEAN,ocean_INLAND,ocean_ISLAND,ocean_NEAR BAY,ocean_NEAR OCEAN,rooms_per_household,bedrooms_per_room,population_per_household,bedrooms_per_household,rooms_per_person
0,-1.423017,1.013801,1.860989,0.311737,1.361482,0.137243,1.394645,-0.936838,458300.0,0.638781,1.326381,0.457605,1.319048,-1.079412,0,0,0,1,0,-1.603562,2.363853,-1.863764,0.003000,0.274803
1,0.596312,-0.702021,0.907456,-0.308794,-0.436386,-0.694028,-0.373691,1.171813,483800.0,-0.064713,-0.253986,-0.716519,-0.145608,1.231907,1,0,0,0,0,0.123497,-0.606128,-1.436105,-0.529295,1.609763
2,-1.203090,1.276331,0.351229,-0.712413,-0.760049,-0.789030,-0.775942,-0.760117,101700.0,-0.863126,-0.926791,-0.947813,-0.921534,-0.793203,0,1,0,0,0,-0.003995,-0.220221,-0.206661,-0.054507,-0.010465


In [19]:
#Dejar listas las matrices para la siguiente etapa
# Versión sin escalar
X_fe = df_fe.drop(columns=[target]).copy()
y_fe = df_fe[target].copy()

# Versión escalada
X_fe_scaled = df_fe_scaled.drop(columns=[target]).copy()

print("X_fe shape:", X_fe.shape)
print("X_fe_scaled shape:", X_fe_scaled.shape)
print("y_fe shape:", y_fe.shape)

X_fe shape: (16509, 23)
X_fe_scaled shape: (16509, 23)
y_fe shape: (16509,)


In [20]:
print("Cantidad final de columnas en df_fe:", df_fe.shape[1])
print("Cantidad final de columnas en df_fe_scaled:", df_fe_scaled.shape[1])

print("\nColumnas finales:")
print(df_fe_scaled.columns.tolist())

Cantidad final de columnas en df_fe: 24
Cantidad final de columnas en df_fe_scaled: 24

Columnas finales:
['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value', 'log_total_rooms', 'log_total_bedrooms', 'log_population', 'log_households', 'log_median_income', 'ocean_<1H OCEAN', 'ocean_INLAND', 'ocean_ISLAND', 'ocean_NEAR BAY', 'ocean_NEAR OCEAN', 'rooms_per_household', 'bedrooms_per_room', 'population_per_household', 'bedrooms_per_household', 'rooms_per_person']
